In [1]:
# Import from main and experiments library
import os
from exp_library import *
from RoBERTa_library import *
os.chdir("../")
from library import *

# filter the warnings for clarity
import warnings
warnings.filterwarnings("ignore")

## Bankruptcy Prediction - Textual Models 💸📜📈

### Introduction

In this notebook, we include the experiments for the next-year bankruptcy prediction task using textual predictors from the following paper:

```Henri Arno, Klaas Mulier, Joke Baeck, and Thomas Demeester. 2023. From Numbers to Words: Multi-Modal Bankruptcy Prediction Using the ECL Dataset. In Proceedings of the Sixth Workshop on Financial Technology and Natural Language Processing (FinNLP), Bali, Indonesia. Association for Computational Linguistics.```

For an explanation of the task, see the notebook *Numerical Models.ipynb*. The figure below shows the task intuitively.

<div>
<img src="../images/task.png" width="800" align="left"/>
</div>

### Read in the data

In [2]:
# Read in the dataset.
dataset = pd.read_csv('ECL.csv', index_col=0)

In [3]:
# Get the subset of labelled 10Ks
prediction_subset = dataset.loc[(dataset['can_label'] == True) & (dataset['qualified'] == 'Yes')].reset_index(drop=True)

### Add textual predictors

#### Pre-process the documents

First, we will store item 7 (the management discussion and analysis section) of each 10K in a separate corpus. For the transformer-based RoBERTa model, we do not preprocess the documents, for the TF-IDF model, we create a clean version. The cleaning process involves several steps: tokenization, lowercasing, lemmatizing and removal of stopwords, numerals and punctuation. We store the raw and the cleaned version of the corpus in the *.data/* directory.

In [4]:
# Path to data
original_corpus = './data'
clean_corpus = './data/clean_corpus'
raw_corpus = './data/raw_corpus'

# Add a flag to check if we still need to clean the documents
clean = True


# Create directories
try:
    os.mkdir(clean_corpus + '/')
    os.mkdir(raw_corpus + '/')
    for i in range(1993,2024):
        os.mkdir(clean_corpus + '/' + str(i))
        os.mkdir(raw_corpus + '/' + str(i))
except:
    print('Corpera already exist')
    clean = False

Corpera already exist


In [5]:
# Check if we still need to create the corpera and clean the documents
if clean:

    # Loop over each row in the prediction_subset DataFrame
    for idx, row in prediction_subset.iterrows():

        # Track progress
        if idx % 1000 == 0:
            print(f"Processing file [{idx}/{len(prediction_subset)}]")

        # Read in the file
        filename = row['filename']
        file_path = original_corpus + filename
        with open(file_path, "r", encoding="utf-8") as f:
            file_data = json.load(f)

        # Extract item 7 and clean
        document = file_data.get('item_7', '')
        tokens = tokenize_lemmatize(document)
        clean_tokens = remove_stop_punct_num(tokens)
        clean_document = ' '.join(clean_tokens)


        # Define file paths for storing
        file_name_without_extension = os.path.splitext(filename)[0]
        preprocessed_filepath = clean_corpus + file_name_without_extension + '.txt'
        raw_filepath = raw_corpus + file_name_without_extension + '.txt'

        # Store the cleaned version
        with open(preprocessed_filepath, "w", encoding="utf-8") as preprocessed_file:
            preprocessed_file.write(clean_document)

        # Store the raw version
        with open(raw_filepath, "w", encoding="utf-8") as raw_file:
            raw_file.write(document)

print("Done")

Done


#### Create train and test set

In [6]:
# Adjust file extension in filename variable
prediction_subset['filename'] = prediction_subset['filename'].str.replace('.json', '.txt')
# Split in train and test
train = prediction_subset.loc[prediction_subset['bankruptcy_prediction_split'] == 'train']
test = prediction_subset.loc[prediction_subset['bankruptcy_prediction_split'] == 'test']

# Models

In the following section, we show how we trained the models, after hyperparameter optimisation, using the sci-kit learn, HuggingFace transformers and the PyTorch frameworks. We do not include the optimisation itself, but immediatly train the models with the optimal hyperparameter settings.

## TF-IDF

In [7]:
# split predictors and labels
train_X = clean_corpus + train['filename']
test_X = clean_corpus + test['filename']

train_y = train['label']
test_y = test['label']

In [8]:
# create the pipeline
TF_IDF = Pipeline([
        ('vect', TfidfVectorizer(input='filename', lowercase=True, 
                                 strip_accents='ascii', stop_words='english', min_df=2, ngram_range = (1,2))),
        ('clf', LogisticRegression(penalty = 'l1', C = 1, class_weight = 'balanced', 
                                   solver='liblinear'))])

# train model
TF_IDF.fit(X=train_X, y= train_y)

# evaluate the model
preds = TF_IDF.predict_proba(test_X)[:, 1]
evaluate(labels=test_y, predictions=preds)

-- RESULTS --
AUC: 0.8855
AP: 0.2387
recall@100: 0.2869
CAP: 0.7711


## RoBERTa

For the RoBERTa transfomer-based model, we use the HuggingFace transformers library in combination with PyTorch. Furthermore, we use a GPU server to speed up the training process (on a CPU, the training time is very long, you can experiment with the code using a small sample of the corpus). Make sure that you have the correct installation of PyTorch for your setup: https://pytorch.org/

#### Use a sample or the entire dataset

In [9]:
# Set this option to True if you want to explore the code with a small, balanced train and test set.
sample = True

if sample:
    
    # sample 5 random positive and 5 random negative 10K's from the train and test set
    pos_train = train.loc[train['label'] == True].sample(5)
    pos_test = test.loc[test['label'] == True].sample(5)
    neg_train = train.loc[train['label'] == False].sample(5)
    neg_test = test.loc[test['label'] == False].sample(5)

    # adjust the train and test set to these small samples
    train = pd.concat([pos_train, neg_train])
    test = pd.concat([pos_test, neg_test])

#### Initialize model and set parameters

In [10]:
# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
# Load pretrained RoBERTa tokenizer and model 
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")
model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)

Some weights of the model checkpoint at roberta-large were not used when initializing RobertaForSequenceClassification: ['lm_head.bias', 'lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias', 'lm_head.decoder.weight', 'roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.weight', 'classif

In [12]:
# Set initial parameters
batch_size = 16 # we use gradient accumulation to simulate larger batch sizes
learning_rate = 2e-3
num_epochs = 2

# Define the optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Create dataset and dataloaders
train_dataset = CustomDataset(train, tokenizer, raw_corpus)
test_dataset = CustomDataset(test, tokenizer, raw_corpus)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

#### Define weighted loss function

In [13]:
# Calculate class weights based on prevalence - only use train distribution
class_counts = train['label'].value_counts().to_dict()
total_samples = len(train)
class_weights = [total_samples / (2 * class_counts[False]), total_samples / (2 * class_counts[True])]

# Define the loss function
loss_fn = nn.CrossEntropyLoss(weight=torch.Tensor(class_weights).to(device))

#### Training

In [14]:
# Freeze the transformer layers for the first epoch
for param in model.roberta.parameters():
    param.requires_grad = False
    
# Set model in train mode
model.to(device)
model.train()

# Track positives seen
positives = 0

# Define the number of gradient accumulation steps
# Based on class distribution, we should see about 1 positive sample per 127 negative ones
accumulation_steps = 20

In [15]:
# Loop over epoch
for epoch in range(num_epochs):
    
    # Track loss per epoch
    total_loss = 0.0

    # Unfreeze the weights in second epoch and adjust the learning rate
    if epoch == 1:
        for param in model.roberta.parameters():
            param.requires_grad = True
            optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)


    # Loop over data in batches
    for idx, batch in enumerate(train_loader):

        # Track progress
        if idx%100==0:
            print(f"Processing batch: {idx} in epoch {epoch} of {int(train_dataset.__len__() / batch_size)} batches")
            print(f"Positives seen: {positives}")

        
        # Put tensors on the correct device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Track the number of positives seen
        positives += int(batch['labels'].sum().cpu())

        # Forwards pass
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        logits = outputs.logits  # Get the raw logits

        # Compute loss
        loss = loss_fn(logits, labels)

        # Adjust the loss for gradient accumulation
        loss = loss / accumulation_steps
        
        # Compute gradients
        loss.backward()

        # Check if we need to update the weights
        if ((idx + 1) % accumulation_steps == 0) or (idx + 1 == len(train_loader)):

            # Perform weight update and reset gradients
            optimizer.step()
            optimizer.zero_grad()
        
        # Add loss to total loss
        total_loss += loss.item()
        # Compute loss per batch 
        loss_batch = loss.item() / batch_size

    # Print final loss after training
    average_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Average Loss: {average_loss:.4f}")

Processing batch: 0 in epoch 0 of 0 batches
Positives seen: 0
Epoch 1, Average Loss: 0.0377
Processing batch: 0 in epoch 1 of 0 batches
Positives seen: 5
Epoch 2, Average Loss: 0.2054


#### Model evaluation

In [16]:
# Evaluation
model.eval()
test_labels = []
test_preds = []

In [17]:
# Test loop
for idx, batch in enumerate(test_loader):
    
    # Track progress
    if idx%100==0:
        print(f"Processing batch {idx} of {int(len(test_dataset) / batch_size)}")
        print(f"Positive samples: {sum(test_labels)}")
    
    # Put tensors on the correct device
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].cpu().numpy()

    # Make predictions
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)

    # Store results
    test_labels.extend(labels)
    test_preds.extend(probabilities.cpu().numpy())

Processing batch 0 of 0
Positive samples: 0


In [20]:
# Evaluate the model
preds = [label[1] for label in test_preds]
if sample:
    print('Results on a small subsample of the data:\n')
evaluate(labels=test_labels, predictions=preds)

Results on a small subsample of the data:

-- RESULTS --
AUC: 0.64
AP: 0.6978
recall@100: 1.0
CAP: 0.28
